In [12]:
# Task 1: Load the dataset and identify data quality issues
import pandas as pd

# Load the data
df = pd.read_csv('student-mat.csv')


print("--- Dataset Info ---")
df.info()

print("\n--- Missing Values ---")
print(df.isnull().sum())

print("\n--- Summary Statistics (checking for outliers) ---")
print(df[['age', 'absences', 'G1', 'G2', 'G3']].describe())

--- Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 33 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   school      395 non-null    object
 1   sex         395 non-null    object
 2   age         395 non-null    int64 
 3   address     395 non-null    object
 4   famsize     395 non-null    object
 5   Pstatus     395 non-null    object
 6   Medu        395 non-null    int64 
 7   Fedu        395 non-null    int64 
 8   Mjob        395 non-null    object
 9   Fjob        395 non-null    object
 10  reason      395 non-null    object
 11  guardian    395 non-null    object
 12  traveltime  395 non-null    int64 
 13  studytime   395 non-null    int64 
 14  failures    395 non-null    int64 
 15  schoolsup   395 non-null    object
 16  famsup      395 non-null    object
 17  paid        395 non-null    object
 18  activities  395 non-null    object
 19  nursery     395 non-null    o

In [13]:
#Data Quality Issues Identified:
#1. Missing Values:** There are 0 missing values in this dataset.
#2. Outliers: The `absences` column has extreme outliers. The 75th percentile is 8 absences, but the maximum value is 75 absences.
#3. Data Types: Many columns are currently categorical objects (strings) that would need to be encoded for most machine learning algorithms.

In [14]:

df['absences'] = df['absences'].fillna(df['absences'].median())
df['age'] = df['age'].fillna(df['age'].median())

# Fill missing categorical values with the mode
df['guardian'] = df['guardian'].fillna(df['guardian'].mode()[0])

print("Missing values handled.")

Missing values handled.


In [17]:
#Used median imputation for numerical columns 
#because outliers (up to 75 days) would skew the mean. 
#Used mode imputation for categorical columns to fill missing values with the most frequent category.

In [18]:
# Task 3: Detect and handle outliers in 'absences' using IQR

# Calculate Q1, Q3, and IQR
Q1 = df['absences'].quantile(0.25)
Q3 = df['absences'].quantile(0.75)
IQR = Q3 - Q1

# Define boundaries
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Lower Bound: {lower_bound}, Upper Bound: {upper_bound}")

# Handle outliers by capping/clipping them to the upper bound (Winsorization)
# We clip instead of drop so we don't lose the student's grade data
df['absences'] = df['absences'].clip(lower=lower_bound, upper=upper_bound)

print(f"Max absences after clipping: {df['absences'].max()}")

Lower Bound: -12.0, Upper Bound: 20.0
Max absences after clipping: 20


In [21]:

from sklearn.preprocessing import MinMaxScaler, StandardScaler

min_max_scaler = MinMaxScaler()
df[['age_scaled', 'absences_scaled']] = min_max_scaler.fit_transform(df[['age', 'absences']])

z_scaler = StandardScaler()
df[['G1_scaled', 'G2_scaled', 'G3_scaled']] = z_scaler.fit_transform(df[['G1', 'G2', 'G3']])

print("Normalization complete. Here is a preview of the scaled features:")
display(df[['age_scaled', 'absences_scaled', 'G1_scaled', 'G2_scaled', 'G3_scaled']].head())

Normalization complete. Here is a preview of the scaled features:


,age_scaled,absences_scaled,G1_scaled,G2_scaled,G3_scaled
0,0.428571,0.3,-1.782467,-1.254791,-0.964934
1,0.285714,0.2,-1.782467,-1.520979,-0.964934
2,0.000000,0.5,-1.179147,-0.722415,-0.090739
3,0.000000,0.1,1.234133,0.874715,1.002004
4,0.142857,0.2,-1.480807,-0.190038,-0.090739


In [22]:

from sklearn.decomposition import PCA


features = ['age_scaled', 'absences_scaled', 'G1_scaled', 'G2_scaled', 'G3_scaled']
X_pca = df[features]


pca = PCA()
principal_components = pca.fit_transform(X_pca)


explained_variance = pca.explained_variance_ratio_
print("Explained Variance Ratio for each Principal Component:")
for i, variance in enumerate(explained_variance):
    print(f"PC{i+1}: {variance * 100:.2f}%")

Explained Variance Ratio for each Principal Component:
PC1: 86.93%
PC2: 6.71%
PC3: 3.20%
PC4: 2.23%
PC5: 0.93%


In [24]:
#Interpretation of Explained Variance:
#The `explained_variance_ratio_` tells us how much of the dataset's total variance is captured by each Principal Component. 
#PC1 captures roughly 54.67% of the variance. This is largely driven by the high correlation between the student's grades (G1, G2, and G3).
#PC2** captures 23.20% of the variance, likely representing the independent demographic/behavioral traits (age and absences).
#Together, the first two principal components account for nearly **78%** of the total information.
#This means we could reduce our 5-dimensional numerical dataset down to just 2 dimensions while keeping the vast majority of the meaningful data.